<a href="https://colab.research.google.com/github/fboldt/aulas-am-bsi/blob/main/aula14a_decision_tree_discrete_atributes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import zipfile
import io

url = "https://cdn.uci-ics-mlr-prod.aws.uci.edu/19/car%2Bevaluation.zip"

response = requests.get(url)
response.raise_for_status() # Raise an exception for HTTP errors

with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
    zf.extractall()
    print(f"Downloaded and extracted {url.split('/')[-1]} successfully.")

Downloaded and extracted car%2Bevaluation.zip successfully.


In [2]:
import numpy as np

data = np.genfromtxt('car.data', delimiter=',', dtype=str)
print(data.shape)
print(data[:5])

(1728, 7)
[['vhigh' 'vhigh' '2' '2' 'small' 'low' 'unacc']
 ['vhigh' 'vhigh' '2' '2' 'small' 'med' 'unacc']
 ['vhigh' 'vhigh' '2' '2' 'small' 'high' 'unacc']
 ['vhigh' 'vhigh' '2' '2' 'med' 'low' 'unacc']
 ['vhigh' 'vhigh' '2' '2' 'med' 'med' 'unacc']]


In [3]:
X, y = data[:,:6], data[:,6]
print(X.shape)
print(y.shape)
print(X[:5])
print(y[:5])

(1728, 6)
(1728,)
[['vhigh' 'vhigh' '2' '2' 'small' 'low']
 ['vhigh' 'vhigh' '2' '2' 'small' 'med']
 ['vhigh' 'vhigh' '2' '2' 'small' 'high']
 ['vhigh' 'vhigh' '2' '2' 'med' 'low']
 ['vhigh' 'vhigh' '2' '2' 'med' 'med']]
['unacc' 'unacc' 'unacc' 'unacc' 'unacc']


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [5]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score
from collections import Counter

def most_common(lst):
  data = Counter(lst)
  return data.most_common(1)[0][0]

class ZeroR(BaseEstimator, ClassifierMixin):
    def fit(self, X, y):
      self.answer = most_common(y)
      return self
    def predict(self, X):
      return [self.answer] * len(X)

model = ZeroR()
model.fit(X, y)
ypred = model.predict(X)
print(accuracy_score(y, ypred))

0.7002314814814815


In [6]:
model.answer

np.str_('unacc')

In [7]:
model = ZeroR()
model.fit(X_train, y_train)
ypred = model.predict(X_test)
print(accuracy_score(y_test, ypred))

0.6994219653179191


In [8]:
import numpy as np

# apenas uma característica
class DecisionTree(BaseEstimator, ClassifierMixin):
  def __init__(self, feature=0):
    self.feature = feature

  def fit(self, X, y):
    self.value = list(set(X[:,self.feature]))[0]
    equals = X[:,self.feature] == self.value
    if sum(equals)==0 or sum(~equals)==0:
      self.answer = most_common(y)
      self.n_samples = len(y)
    else:
      self.equals = DecisionTree(self.feature).fit(X[equals], y[equals])
      self.not_equals = DecisionTree(self.feature).fit(X[~equals], y[~equals])
    return self

  def predict(self, X):
    if hasattr(self, 'answer'):
      return [self.answer] * len(X)
    else:
      equals = X[:,self.feature] == self.value
    return np.where(equals, self.equals.predict(X), self.not_equals.predict(X))

model = DecisionTree()
model.fit(X_train, y_train)
ypred = model.predict(X_test)
print(accuracy_score(y_test, ypred))

0.6994219653179191


In [9]:
def print_tree(tree, depth=0):
  if hasattr(tree, 'answer'):
    print('\t' * depth, f"answer: {tree.answer} ({tree.n_samples})")
  else:
    print('\t' * depth, f"value: {tree.value}")
    print_tree(tree.equals, depth+1)
    print_tree(tree.not_equals, depth+1)

print_tree(model)

 value: low
	 answer: unacc (336)
	 value: vhigh
		 answer: unacc (357)
		 value: med
			 answer: unacc (330)
			 answer: unacc (359)


In [10]:
def gini(y):
  label = np.unique(y)
  aux = 0
  for l in label:
    p = np.sum(y==l)/len(y)
    aux += p**2
  return 1-aux

print(gini(y_train))

0.4570454112310227


In [11]:
print(gini(np.ones(10)))

0.0


In [12]:
def impurity_value(y, x, value, impurity_function=gini):
  equals = x == value
  imp_equals = impurity_function(y[equals])
  imp_not_equals = impurity_function(y[~equals])
  return len(y[equals])/len(y)*imp_equals + len(y[~equals])/len(y)*imp_not_equals

print(impurity_value(y_train, X_train[:,0], 'vhigh'))

0.4482135562918824


In [13]:
def best_split(y, x, impurity_function=gini):
  best_value = None
  best_impurity = 1
  for value in set(x):
    impurity = impurity_value(y, x, value, impurity_function)
    if impurity < best_impurity:
      best_impurity = impurity
      best_value = value
  return best_value, best_impurity

print(best_split(y_train, X_train[:,2]))

(np.str_('5more'), np.float64(0.4563455711991396))


In [14]:
def best_feature(y, X, impurity_function=gini):
  best_feature = None
  best_value = None
  best_impurity = 1
  for feature in range(X.shape[1]):
    value, impurity = best_split(y, X[:,feature], impurity_function)
    if impurity < best_impurity:
      best_impurity = impurity
      best_feature = feature
      best_value = value
  return best_feature, best_value, best_impurity

print(best_feature(y_train, X_train))

(5, np.str_('low'), np.float64(0.38499511557696947))


In [16]:
class DecisionTree(BaseEstimator, ClassifierMixin):
  def fit(self, X, y):
    self.feature, self.value, self.impurity = best_feature(y, X)
    equals = X[:,self.feature] == self.value
    if sum(equals)==0 or sum(~equals)==0:
      self.answer = most_common(y)
      self.n_samples = len(y)
    else:
      self.equals = DecisionTree().fit(X[equals], y[equals])
      self.not_equals = DecisionTree().fit(X[~equals], y[~equals])
    return self

  def predict(self, X):
    if hasattr(self, 'answer'):
      return [self.answer] * len(X)
    else:
      equals = X[:,self.feature] == self.value
    return np.where(equals, self.equals.predict(X), self.not_equals.predict(X))

model = DecisionTree()
model.fit(X_train, y_train)
ypred = model.predict(X_test)
print(accuracy_score(y_test, ypred))

0.9710982658959537


In [17]:
print_tree(model)

 value: low
	 value: low
		 answer: unacc (112)
		 value: vhigh
			 answer: unacc (124)
			 value: med
				 answer: unacc (106)
				 answer: unacc (123)
	 value: 2
		 value: low
			 answer: unacc (74)
			 value: vhigh
				 answer: unacc (78)
				 value: med
					 answer: unacc (71)
					 answer: unacc (79)
		 value: vhigh
			 value: med
				 value: small
					 value: med
						 answer: unacc (6)
						 value: 2
							 value: 4
								 answer: acc (1)
								 answer: unacc (1)
							 answer: acc (5)
					 value: big
						 answer: acc (14)
						 value: 2
							 value: 4
								 answer: unacc (1)
								 answer: acc (1)
							 value: 3
								 value: 4
									 value: med
										 answer: unacc (1)
										 answer: acc (1)
									 answer: acc (1)
								 answer: acc (8)
				 value: low
					 value: med
						 value: big
							 answer: acc (7)
							 value: 5more
								 value: small
									 answer: unacc (1)
									 answer: acc (2)
								 answer: unacc (8)
				

In [18]:
from sklearn.model_selection import cross_validate, KFold

model = DecisionTree()
scores = cross_validate(model, X, y,
                        cv=KFold(shuffle=True))
print(scores['test_score'])
print(np.mean(scores['test_score']))

[0.97398844 0.96820809 0.97687861 0.97101449 0.96811594]
0.9716411158582557
